# CURC VIIRS SNPP VNP09GA First-Run Workflow

This notebook illustrates the CURC-specific workflow layer as a thin interactive front end over `workflows/curc`.

The intended user model is:
- choose a sensor, platform, tile set, and water year,
- inspect discovered inputs and planned workflow steps,
- run one step at a time or eventually submit distributed Slurm jobs.

This notebook is set up for a first full-water-year `VNP09GA` inversion run on tiles `h08v04`, `h08v05`, `h09v04`, `h09v05`, and `h10v04` for water year `2024` (`2023-10-01` through `2024-09-30`).

After this SNPP pass is validated, the same sequence can be reused for `VJ109GA` and `VJ209GA` by swapping the source root and platform defaults.

In [1]:
from __future__ import annotations

from dataclasses import asdict
import json
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "spires").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"repo_root={REPO_ROOT}")

from spires.sensors.registry import list_supported_sensor_platforms
from workflows.curc.config import CurcWorkflowConfig, SlurmProfile
from workflows.curc.discovery import discover_viirs_snpp_water_year_reflectance_files
from workflows.curc.dates import default_r0_year_for_water_year
from workflows.curc.paths import ancillary_dir, log_root, output_raw_water_year_root, reflectance_dir, r0_dir
from workflows.curc.runner import plan_viirs_snpp_steps


def print_step_summary(step) -> None:
    print(
        json.dumps(
            {
                "step": step.step,
                "tile": step.tile,
                "date_count": step.date_count,
                "source_path_count": len(step.source_paths),
                "destination_path": step.destination_path,
                "r0_year": step.r0_year,
            },
            indent=2,
        )
    )


def print_payload_summary(payload: dict[str, object]) -> None:
    print(
        json.dumps(
            {
                "step": payload.get("step"),
                "tile": payload.get("tile"),
                "water_year": payload.get("water_year"),
                "date_count": payload.get("date_count"),
                "source_path_count": payload.get("source_path_count"),
                "destination_path": payload.get("destination_path"),
                "r0_year": payload.get("r0_year"),
                "mode": payload.get("mode"),
                "command_count": len(payload.get("commands", [])),
                "expected_static_file_count": len(payload.get("expected_static_files", [])),
                "output_dataset_path": payload.get("output_dataset_path"),
                "zarr_path": payload.get("zarr_path"),
                "chunks": payload.get("chunks"),
                "executed": payload.get("executed"),
            },
            indent=2,
        )
    )


def print_array_payload_summary(payload: dict[str, object], tile: str) -> None:
    print(
        json.dumps(
            {
                "tile": tile,
                "manifest_path": payload.get("manifest_path"),
                "job_name": payload.get("job_name"),
                "task_count": payload.get("task_count"),
                "max_concurrent_tasks": payload.get("max_concurrent_tasks"),
                "output_root": payload.get("output_root"),
            },
            indent=2,
        )
    )


def print_submit_result_summary(result: dict[str, object], tile: str) -> None:
    print(
        json.dumps(
            {
                "tile": tile,
                "submitted": result.get("submitted"),
                "sbatch_stdout": result.get("sbatch_stdout"),
                "manifest_path": result.get("manifest_path"),
            },
            indent=2,
        )
    )


def print_scan_result_summary(result: dict[str, object], tile: str) -> None:
    print(
        json.dumps(
            {
                "tile": tile,
                "manifest_path": result.get("manifest_path"),
                "status_counts": result.get("status_counts"),
                "missing_summary_count": len(result.get("missing_summary", [])),
                "retry_eligible_count": len(result.get("retry_eligible", [])),
            },
            indent=2,
        )
    )


repo_root=/projects/ropa5718/SpiPy


## Configuration

The notebook should not define separate workflow logic, but it should expose the user-editable parameters clearly. These defaults are set for the first full `VNP09GA` WY2024 runthrough.

In [2]:
supported = list_supported_sensor_platforms()
print(json.dumps(supported, indent=2))
print()

# User-editable workflow parameters.
SCRATCH_ROOT = Path("/scratch/alpine/ropa5718/spipy")
INPUT_SOURCE_ROOT = Path("/pl/active/rittger_ops/vnp09ga.002")
SENSOR = "viirs"
PLATFORMS = ("snpp",)
TILES = ("h08v04", "h08v05", "h09v04", "h09v05", "h10v04")
WATER_YEARS = (2024,)
DATES = ()
DATE_GLOB = "*"
DRY_RUN = False
APPLY_VALID_INVERSION_MASK = False
SLURM_EXCLUDE_NODES = (
    "bgpu-biokem1,bgpu-biokem2,bgpu-biokem3,bgpu-bortz1,bgpu-curc2,bgpu-curc4,"
    "bgpu-g4-18,bgpu-g4-u20,bgpu-g4-u24,bgpu-g6-u20,bgpu-ivc,bhpc-c5-u31-1,"
    "blanca-g4-u14-3,bmem-rico1,bnode0108,bgpu-g6-u34,bhpc-c5-u7-12,bnode0414,"
    "bnode0508,bhpc-c7-u7-8,bgpu-g6-u25,bgpu-chbe-rdi1"
)
SLURM_EXCLUDE_ARG = f"--exclude={SLURM_EXCLUDE_NODES}"
SLURM_MEM = "8G"

config = CurcWorkflowConfig(
    scratch_root=SCRATCH_ROOT,
    input_source_root=INPUT_SOURCE_ROOT,
    sensor=SENSOR,
    platforms=PLATFORMS,
    tiles=TILES,
    years=(),
    water_years=WATER_YEARS,
    dates=DATES,
    date_glob=DATE_GLOB,
    dry_run=DRY_RUN,
    apply_valid_inversion_mask=APPLY_VALID_INVERSION_MASK,
    slurm_profile=SlurmProfile(mem=SLURM_MEM, extra_args=(SLURM_EXCLUDE_ARG,)),
).canonicalized()

config


{
  "modis": [
    "terra",
    "aqua"
  ],
  "viirs": [
    "snpp",
    "noaa20",
    "noaa21"
  ]
}



CurcWorkflowConfig(scratch_root=PosixPath('/scratch/alpine/ropa5718/spipy'), input_source_root=PosixPath('/pl/active/rittger_ops/vnp09ga.002'), sensor='viirs', platforms=('snpp',), tiles=('h08v04', 'h08v05', 'h09v04', 'h09v05', 'h10v04'), years=(), water_years=(2024,), dates=(), date_glob='*', dry_run=False, max_auto_retry_count=3, apply_valid_inversion_mask=False, use_grouping=True, grouping_method='chunk_bin_mean', slurm_profile=SlurmProfile(account=None, qos=None, time=None, mem='8G', cpus_per_task=None, output_dir=None, extra_args=('--exclude=bgpu-biokem1,bgpu-biokem2,bgpu-biokem3,bgpu-bortz1,bgpu-curc2,bgpu-curc4,bgpu-g4-18,bgpu-g4-u20,bgpu-g4-u24,bgpu-g6-u20,bgpu-ivc,bhpc-c5-u31-1,blanca-g4-u14-3,bmem-rico1,bnode0108,bgpu-g6-u34,bhpc-c5-u7-12,bnode0414,bnode0508,bhpc-c7-u7-8,bgpu-g6-u25,bgpu-chbe-rdi1',)))

## Scratch Layout

The CURC workflow keeps shared staged inputs under `/scratch/alpine/.../input` and dated inversion outputs under `/scratch/alpine/.../output`. For a WY2024 run, the default R0 year is `2023`.

In [3]:
platform = config.platforms[0]
tiles = config.tiles
water_year = config.water_years[0]
r0_year = default_r0_year_for_water_year(water_year)

for tile in tiles:
    print(f"\n[{tile}]")
    print("reflectance:", reflectance_dir(config, platform) / tile / str(water_year))
    print("ancillary:", ancillary_dir(config, platform) / tile)
    print("r0:", r0_dir(config, platform) / tile / str(r0_year))
    print("output_raw_wy:", output_raw_water_year_root(config, platform, tile, water_year))
print("logs:", log_root(config))



[h08v04]
reflectance: /scratch/alpine/ropa5718/spipy/input/viirs/snpp/reflectance/h08v04/2024
ancillary: /scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/h08v04
r0: /scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/r0/h08v04/2023
output_raw_wy: /scratch/alpine/ropa5718/spipy/output/viirs/snpp/h08v04/raw/wy2024

[h08v05]
reflectance: /scratch/alpine/ropa5718/spipy/input/viirs/snpp/reflectance/h08v05/2024
ancillary: /scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/h08v05
r0: /scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/r0/h08v05/2023
output_raw_wy: /scratch/alpine/ropa5718/spipy/output/viirs/snpp/h08v05/raw/wy2024

[h09v04]
reflectance: /scratch/alpine/ropa5718/spipy/input/viirs/snpp/reflectance/h09v04/2024
ancillary: /scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/h09v04
r0: /scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/r0/h09v04/2023
output_raw_wy: /scratch/alpine/ropa5718/spipy/output/viirs/snpp/h09v04/raw/wy2024

[h09v05]


## Full Water-Year Discovery

This discovery step is the first concrete replacement for the opaque batch bash scripts. It gives a direct view of which source files are available for each tile in the requested water year.

In [4]:
reflectance_files_by_tile = {}

for tile in tiles:
    reflectance_files = discover_viirs_snpp_water_year_reflectance_files(
        config,
        tile=tile,
        water_year=water_year,
    )
    reflectance_files_by_tile[tile] = reflectance_files
    print(f"\n[{tile}]")
    print("water_year_file_count", len(reflectance_files))
    print("first_file", reflectance_files[0] if reflectance_files else None)
    print("last_file", reflectance_files[-1] if reflectance_files else None)



[h08v04]
water_year_file_count 348
first_file /pl/active/rittger_ops/vnp09ga.002/input/h08v04/2023/VNP09GA.A2023274.h08v04.002.2023276013004.h5
last_file /pl/active/rittger_ops/vnp09ga.002/input/h08v04/2024/VNP09GA.A2024274.h08v04.002.2024275232344.h5

[h08v05]
water_year_file_count 348
first_file /pl/active/rittger_ops/vnp09ga.002/input/h08v05/2023/VNP09GA.A2023274.h08v05.002.2023276013043.h5
last_file /pl/active/rittger_ops/vnp09ga.002/input/h08v05/2024/VNP09GA.A2024274.h08v05.002.2024275232407.h5

[h09v04]
water_year_file_count 348
first_file /pl/active/rittger_ops/vnp09ga.002/input/h09v04/2023/VNP09GA.A2023274.h09v04.002.2023276013029.h5
last_file /pl/active/rittger_ops/vnp09ga.002/input/h09v04/2024/VNP09GA.A2024274.h09v04.002.2024275232516.h5

[h09v05]
water_year_file_count 348
first_file /pl/active/rittger_ops/vnp09ga.002/input/h09v05/2023/VNP09GA.A2023274.h09v05.002.2023276013008.h5
last_file /pl/active/rittger_ops/vnp09ga.002/input/h09v05/2024/VNP09GA.A2024274.h09v05.002.20242

## Full Water-Year Step Plan

The workflow should be step-oriented rather than all-or-nothing. For a full water year, the planner expands each tile request into the major steps that will later become executable or submitted jobs.

In [5]:
full_water_year_steps_by_tile = {}
planned_r0_year_by_tile = {}

for tile in tiles:
    full_water_year_steps = plan_viirs_snpp_steps(
        config,
        tile=tile,
        water_year=water_year,
    )
    full_water_year_steps_by_tile[tile] = full_water_year_steps
    step_by_name = {step.step: step for step in full_water_year_steps}
    planned_r0_year_by_tile[tile] = step_by_name["build_r0"].r0_year

    print(f"\n[{tile}]")
    for step in full_water_year_steps:
        print_step_summary(step)



[h08v04]
{
  "step": "stage_reflectance",
  "date_count": 348,
  "destination_path": "/scratch/alpine/ropa5718/spipy/input/viirs/snpp/reflectance/h08v04/2024",
  "r0_year": null
}
{
  "step": "stage_ancillary",
  "date_count": 348,
  "destination_path": "/scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/h08v04",
  "r0_year": null
}
{
  "step": "build_r0",
  "date_count": 122,
  "destination_path": "/scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/r0/h08v04/2023",
  "r0_year": 2023
}
{
  "step": "run_inversion",
  "date_count": 348,
  "destination_path": "/scratch/alpine/ropa5718/spipy/output/viirs/snpp/h08v04/raw/wy2024",
  "r0_year": 2023
}

[h08v05]
{
  "step": "stage_reflectance",
  "date_count": 348,
  "destination_path": "/scratch/alpine/ropa5718/spipy/input/viirs/snpp/reflectance/h08v05/2024",
  "r0_year": null
}
{
  "step": "stage_ancillary",
  "date_count": 348,
  "destination_path": "/scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/h08v05",
  "r0_yea

## Single-Date Rerun Plan

A key design goal is that a user can rerun a single date without relaunching the entire water year. Leave `target_dates` empty during the first full run, then come back here only when you need a narrow rerun.

In [7]:
# Leave empty for the first full-water-year run. Set one or more ISO dates only for targeted reruns.
target_dates = ()

if not target_dates:
    print("Set target_dates to one or more ISO dates before running the rerun planner.")
else:
    for tile in tiles:
        target_date_steps = plan_viirs_snpp_steps(
            config,
            tile=tile,
            water_year=water_year,
            target_dates=target_dates,
        )

        print(f"\n[{tile}]")
        for step in target_date_steps:
            print_step_summary(step)


Set target_dates to one or more ISO dates before running the rerun planner.


## First-Run Order

For a first full WY2024 run, use the notebook in this order:
- preview discovery and the full step plan,
- execute `stage_reflectance`,
- execute `stage_ancillary`,
- execute `build_r0` for the planned R0 year,
- create and submit one full `run_inversion` Slurm array per tile,
- scan the manifest and inspect the log directory after the array finishes.

## Step Execution Helpers

Preview the three direct prerequisite steps before the Slurm submission path. `run_inversion` itself still uses the manifest-backed array submission flow. The helper cells below print compact summaries instead of full per-date payloads.


In [8]:
from workflows.curc.runner import preview_viirs_snpp_step_execution, run_viirs_snpp_step

R0_USE_ZARR = True
R0_ZARR_CHUNKS = {"time": 1, "y": 256, "x": 256, "band": -1}
R0_NDVI_TIE_EPSILON = 0.02

preview_payloads_by_tile = {}

for tile in tiles:
    preview_payloads_by_tile[tile] = {}
    print(f"\n=== {tile} ===")
    for step_name in ("stage_reflectance", "stage_ancillary", "build_r0"):
        step_zarr_path = None
        step_chunks = None
        if step_name == "build_r0" and R0_USE_ZARR:
            step_zarr_path = config.scratch_root / "tmp" / "r0_zarr" / f"{tile}_r0_wy{water_year}.zarr"
            step_chunks = R0_ZARR_CHUNKS
        payload = preview_viirs_snpp_step_execution(
            config,
            tile=tile,
            water_year=water_year,
            step=step_name,
            target_dates=target_dates,
            r0_year=planned_r0_year_by_tile[tile],
            ndvi_tie_epsilon=R0_NDVI_TIE_EPSILON,
            zarr_path=None if step_zarr_path is None else str(step_zarr_path),
            chunks=step_chunks,
        )
        preview_payloads_by_tile[tile][step_name] = payload
        print(f"\n[{step_name}]")
        print_payload_summary(payload)



=== h08v04 ===

[stage_reflectance]
{
  "step": "stage_reflectance",
  "sensor": "viirs",
  "platform": "snpp",
  "tile": "h08v04",
  "water_year": 2024,
  "date_count": 348,
  "dates": [
    "2023-10-01",
    "2023-10-02",
    "2023-10-03",
    "2023-10-04",
    "2023-10-05",
    "2023-10-06",
    "2023-10-07",
    "2023-10-08",
    "2023-10-09",
    "2023-10-10",
    "2023-10-11",
    "2023-10-12",
    "2023-10-13",
    "2023-10-14",
    "2023-10-15",
    "2023-10-16",
    "2023-10-17",
    "2023-10-18",
    "2023-10-19",
    "2023-10-20",
    "2023-10-21",
    "2023-10-22",
    "2023-10-23",
    "2023-10-24",
    "2023-10-25",
    "2023-10-26",
    "2023-10-27",
    "2023-10-28",
    "2023-10-29",
    "2023-10-30",
    "2023-10-31",
    "2023-11-02",
    "2023-11-03",
    "2023-11-04",
    "2023-11-05",
    "2023-11-06",
    "2023-11-07",
    "2023-11-08",
    "2023-11-09",
    "2023-11-10",
    "2023-11-11",
    "2023-11-12",
    "2023-11-13",
    "2023-11-14",
    "2023-11-15",
 

In [9]:
# Flip these to True one step at a time during the first real run.
EXECUTE_STAGE_REFLECTANCE = False
EXECUTE_STAGE_ANCILLARY = False
EXECUTE_BUILD_R0 = False
R0_BUILD_TILES = tuple(tiles)
R0_BUILD_USE_ZARR = True
R0_BUILD_CHUNKS = {"time": 1, "y": 256, "x": 256, "band": -1}
R0_NDVI_TIE_EPSILON = 0.02

step_execution_flags = {
    "stage_reflectance": EXECUTE_STAGE_REFLECTANCE,
    "stage_ancillary": EXECUTE_STAGE_ANCILLARY,
    "build_r0": EXECUTE_BUILD_R0,
}

step_results_by_tile = {}

for tile in tiles:
    step_results_by_tile[tile] = {}
    print(f"\n=== {tile} ===")
    for step_name, execute_step in step_execution_flags.items():
        if step_name == "build_r0" and tile not in R0_BUILD_TILES:
            print(f"\n[{step_name}] skipped; R0_BUILD_TILES={R0_BUILD_TILES}")
            continue
        step_zarr_path = None
        step_chunks = None
        if step_name == "build_r0" and R0_BUILD_USE_ZARR:
            step_zarr_path = config.scratch_root / "tmp" / "r0_zarr" / f"{tile}_r0_wy{water_year}.zarr"
            step_chunks = R0_BUILD_CHUNKS
        result = run_viirs_snpp_step(
            config,
            tile=tile,
            water_year=water_year,
            step=step_name,
            target_dates=target_dates,
            r0_year=planned_r0_year_by_tile[tile],
            execute=execute_step,
            ndvi_tie_epsilon=R0_NDVI_TIE_EPSILON,
            zarr_path=None if step_zarr_path is None else str(step_zarr_path),
            chunks=step_chunks,
            overwrite=False,
            show_progress=True,
        )
        step_results_by_tile[tile][step_name] = result
        print(f"\n[{step_name}]")
        print_payload_summary(result)



=== h08v04 ===

[stage_reflectance]
{
  "step": "stage_reflectance",
  "sensor": "viirs",
  "platform": "snpp",
  "tile": "h08v04",
  "water_year": 2024,
  "date_count": 348,
  "dates": [
    "2023-10-01",
    "2023-10-02",
    "2023-10-03",
    "2023-10-04",
    "2023-10-05",
    "2023-10-06",
    "2023-10-07",
    "2023-10-08",
    "2023-10-09",
    "2023-10-10",
    "2023-10-11",
    "2023-10-12",
    "2023-10-13",
    "2023-10-14",
    "2023-10-15",
    "2023-10-16",
    "2023-10-17",
    "2023-10-18",
    "2023-10-19",
    "2023-10-20",
    "2023-10-21",
    "2023-10-22",
    "2023-10-23",
    "2023-10-24",
    "2023-10-25",
    "2023-10-26",
    "2023-10-27",
    "2023-10-28",
    "2023-10-29",
    "2023-10-30",
    "2023-10-31",
    "2023-11-02",
    "2023-11-03",
    "2023-11-04",
    "2023-11-05",
    "2023-11-06",
    "2023-11-07",
    "2023-11-08",
    "2023-11-09",
    "2023-11-10",
    "2023-11-11",
    "2023-11-12",
    "2023-11-13",
    "2023-11-14",
    "2023-11-15",
 

Building VIIRS R0:   0%|          | 0/122 [00:00<?, ?it/s]

: 

## Slurm Inversion Submission (Full Water Year)

Use these cells to create one full-water-year manifest per tile, submit one inversion array per tile, and inspect scanner output/log files.

Before submitting, make sure `stage_reflectance`, `stage_ancillary`, and `build_r0` have all been executed successfully for this WY2024 run.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

from workflows.curc.runner import plan_viirs_snpp_inversion_array_submission

repo_root = Path.cwd()
if not (repo_root / "scripts" / "submit_curc_inversion_array.py").exists():
    repo_root = repo_root.parent
submission_dates = ()
target_max_concurrent_tasks = None

array_payloads_by_tile = {}

for tile in tiles:
    array_payload = plan_viirs_snpp_inversion_array_submission(
        config,
        tile=tile,
        water_year=water_year,
        target_dates=submission_dates,
        r0_year=planned_r0_year_by_tile[tile],
        max_concurrent_tasks=target_max_concurrent_tasks,
    )
    array_payloads_by_tile[tile] = array_payload

    print(f"\n[{tile}]")
    print_array_payload_summary(array_payload, tile)


In [ ]:
submit_results_by_tile = {}

for tile, array_payload in array_payloads_by_tile.items():
    submit_cmd = [
        sys.executable,
        str(repo_root / "scripts" / "submit_curc_inversion_array.py"),
        array_payload["manifest_path"],
        "--submit",
        "--python-exec",
        sys.executable,
        "--execution-profile",
        "cluster",
    ]

    print(f"\n[{tile}]")
    print("Running:", " ".join(submit_cmd))
    submit_completed = subprocess.run(submit_cmd, check=True, text=True, capture_output=True)
    submit_result = json.loads(submit_completed.stdout)
    submit_results_by_tile[tile] = submit_result
    print_submit_result_summary(submit_result, tile)

    if submit_result.get("submitted"):
        print("Submitted array job:", submit_result.get("sbatch_stdout", "").strip())


In [ ]:
for tile, submit_result in submit_results_by_tile.items():
    submitted_job_id = (submit_result.get("sbatch_stdout", "") or "").strip().split(";")[0]
    print(f"\n[{tile}]")
    if submitted_job_id:
        sacct_cmd = [
            "sacct",
            "-j",
            submitted_job_id,
            "--format=JobID,JobName%35,State,ExitCode,Elapsed,MaxRSS,ReqMem,NodeList",
            "-P",
        ]
        print("Running:", " ".join(sacct_cmd))
        sacct_completed = subprocess.run(sacct_cmd, check=False, text=True, capture_output=True)
        print(sacct_completed.stdout)
        if sacct_completed.stderr.strip():
            print("stderr:\n", sacct_completed.stderr)
    else:
        print("No submitted job id found in submit_results_by_tile[tile]['sbatch_stdout']")


In [ ]:
scan_results_by_tile = {}

for tile, array_payload in array_payloads_by_tile.items():
    scan_cmd = [
        sys.executable,
        str(repo_root / "scripts" / "scan_curc_inversion_array.py"),
        array_payload["manifest_path"],
    ]

    print(f"\n[{tile}]")
    print("Running:", " ".join(scan_cmd))
    scan_completed = subprocess.run(scan_cmd, check=True, text=True, capture_output=True)
    scan_result = json.loads(scan_completed.stdout)
    scan_results_by_tile[tile] = scan_result
    print_scan_result_summary(scan_result, tile)


In [ ]:
log_root = config.scratch_root / "logs"

print("Top-level log root:", log_root)
for path in sorted(log_root.glob("run_inversion*")):
    print(path)

for tile, array_payload in array_payloads_by_tile.items():
    manifest_dir = Path(array_payload["manifest_path"]).parent
    print(f"\nManifest-side files for {tile}: {manifest_dir}")
    for path in sorted(manifest_dir.glob("*inversion*")):
        print(path)
